In [ ]:
import dsautils.calstatus as cs
from dsautils.dsa_store import DsaStore
from astropy.time import Time
import time
import datetime
import yaml
from dsacalib.weights import average_beamformer_solutions
import glob
import os
import numpy as np
from pkg_resources import resource_filename
import astropy.units as u
from dsautils import cnf
from dsacalib.plotting import summary_plot, plot_current_beamformer_solutions
from dsacalib.plotting import plot_beamformer_weights
from dsacalib.routines import get_files_for_cal, calibrate_measurement_set
from dsacalib.weights import get_good_solution, write_beamformer_solutions
from dsacalib.ms_io import convert_calibrator_pass_to_ms, uvh5_to_ms
import matplotlib.pyplot as plt
%matplotlib inline
from matplotlib.backends.backend_pdf import PdfPages
import h5py
myconf = cnf.Conf()


## Creating measurement set from hdf5 files

In [ ]:
# The calibrator pass you want
date = '2024-03-28'
calname = '0319+415'
dec = '+041p5' # basically where the calibrator info is located
duration = 60*u.min # set according to the max length of the ms you want

# edit according to where the hdf5 files are, and where you want the ms to go
msdir = '/operations/calibration/'
hdf5dir = '/operations/correlator/'

# Parameters that don't need to be changed
calsources = resource_filename(
    'dsacalib',
    f'data/calibrator_sources_dec{dec}.csv'
)
refcorr = 'corr03'
filelength = 5*u.min # keep fixed
date_specifier = '{0}*'.format(date)
msname = '{0}/{1}_{2}'.format(msdir, date, calname)
print(calsources)

In [ ]:
# Get a list of the files for each calibrator
filenames = get_files_for_cal(
    calsources,
    hdf5dir,
    'sb01',
    duration,
    filelength,
    date_specifier
)
#print(filenames)
print(filenames[date][calname])

In [ ]:
# manually make ms
# ideally keep refmjd the same for all measurement sets
cal = filenames[date][calname]['cal']
convert_calibrator_pass_to_ms(cal,date,filenames[date][calname]['files'],msdir=msdir,hdf5dir=hdf5dir,refmjd=58849.)